# 0. Imports

In [1]:
import pandas as pd
import numpy as np
import gurobipy as gp
import random

# 1. Data exploration

The initial MILP formulation of the problem is given by:

$$\min\sum_{k\in K}\sum_{i\in N}\sum_{j\in N}b_1 c^kt_{ij}x_{ij}^k+\sum_{i\in P}b_2c^0(1-\sum_{k\in K}\sum_{j\in P\cup D}x_{ij}^k)+b_3E[Q(x,w,a,q,\xi)]$$

In this formulation, $x_{ij}^k$ represents the decision variable (indicator whether arc $ij$ is travelled by vehicle $k$). To start the problem, the travel times between nodes will have to be extracted and associated with a travel time.

## 1.1 Travel times 15-17:

In [427]:
travel_times_15_17 = pd.read_csv("travel_times_15_17.csv", index_col=0)

In [428]:
travel_times_15_17.head(15)

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589152,"Fribourg, Mon-Repos",46.806711,7.172136,270,35,0.58,42,0.70,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589138,"Fribourg, Cité-Jardins",46.809385,7.170446,659,86,1.43,117,1.95,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.30,174,2.90,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8587255,"Fribourg, Tilleul/Cathédrale",46.806090,7.161261,3788,445,7.42,506,8.43,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589161,"Fribourg, St-Pierre",46.803911,7.155266,4335,561,9.35,622,10.37,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8592374,"Fribourg/Freiburg, Pl. Gare",46.802898,7.151410,4719,695,11.58,749,12.48,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589130,"Villars-sur-Glâne, Méridienne",46.794173,7.111828,9363,808,13.47,866,14.43,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589131,"Villars-sur-Glâne, Moncor",46.798570,7.120788,8377,699,11.65,754,12.57,OK
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8588344,"Villars-sur-Glâne, Belle-Croix",46.800233,7.125455,8261,722,12.03,757,12.62,OK


In [429]:
origins_15_17 = pd.unique(travel_times_15_17["origin_name"])
origins_15_17

<StringArray>
[              'Fribourg, Chaley',            'Fribourg, Mon-Repos',
         'Fribourg, Cité-Jardins',             'Fribourg, Boschung',
   'Fribourg, Tilleul/Cathédrale',            'Fribourg, St-Pierre',
    'Fribourg/Freiburg, Pl. Gare',  'Villars-sur-Glâne, Méridienne',
      'Villars-sur-Glâne, Moncor', 'Villars-sur-Glâne, Belle-Croix',
 'Villars-sur-Glâne,Villars-Vert',             'Fribourg, Bertigny',
             'Fribourg, Bellevue',     'Fribourg, Schönberg Dunant',
             'Fribourg, Guintzet', 'Villars-sur-Glâne,Jean Paul II',
  'Villars-sur-Glâne, Hôp. cant.',       'Fribourg, Route-de-Tavel',
              'Fribourg, Kessler',            'Fribourg, Ploetscha',
               'Fribourg, Windig',      'Fribourg, Pont-Zaehringen',
           'Fribourg, Charmettes',            'Fribourg, Industrie',
              'Fribourg, J. Vogt',                'Fribourg, Fries',
              'Fribourg, Gambach',               'Fribourg, Vuille',
          'Givisiez,

In [430]:
print(travel_times_15_17['origin_name'].value_counts())
print(travel_times_15_17['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  196
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196


In [431]:
print(travel_times_15_17["dest_name"].value_counts())
print(travel_times_15_17["dest_name"].value_counts().unique())

dest_name
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Fribourg, Chaley                  196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196
Fr

In [432]:
print(travel_times_15_17.value_counts(['origin_name', 'dest_name']))
print(travel_times_15_17.value_counts(['origin_name', 'dest_name']).unique())

origin_name            dest_name                   
Fribourg, Chaley       Fribourg, Mon-Repos             7
                       Fribourg, Cité-Jardins          7
                       Fribourg, Boschung              7
                       Fribourg, Tilleul/Cathédrale    7
                       Fribourg, St-Pierre             7
                                                      ..
Givisiez, Mont Carmel  Fribourg, Industrie             7
                       Fribourg, J. Vogt               7
                       Fribourg, Fries                 7
                       Fribourg, Gambach               7
                       Fribourg, Vuille                7
Name: count, Length: 812, dtype: int64
[7]


In [433]:
o_d_example_15_17 = travel_times_15_17[
    (travel_times_15_17['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_15_17['dest_name'] == 'Fribourg, Boschung')
]
o_d_example_15_17

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,174,2.90,OK
2026-02-12 15:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,176,2.93,OK
2026-02-12 15:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,163,2.72,OK
2026-02-12 16:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 16:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,155,2.58,OK
2026-02-12 16:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,160,2.67,OK
2026-02-12 17:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK


In [434]:
o_d_example_15_17 = travel_times_15_17[
    (travel_times_15_17['origin_name'] == 'Fribourg, Boschung') &
    (travel_times_15_17['dest_name'] == 'Fribourg, Chaley')
]
o_d_example_15_17

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 15:00:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,162,2.70,OK
2026-02-12 15:20:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,160,2.67,OK
2026-02-12 15:40:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,155,2.58,OK
2026-02-12 16:00:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,152,2.53,OK
2026-02-12 16:20:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,153,2.55,OK
2026-02-12 16:40:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,167,2.78,OK
2026-02-12 17:00:00,8591766,"Fribourg, Boschung",46.811451,7.171016,8589141,"Fribourg, Chaley",46.806281,7.175601,991,138,2.3,163,2.72,OK


## 1.2 Travel times 17-19:

In [435]:
travel_times_17_19 = pd.read_csv("travel_times_17_19.csv", index_col=0)

In [436]:
origins_17_19 = pd.unique(travel_times_17_19["origin_name"])
print(origins_17_19)

<StringArray>
[              'Fribourg, Chaley',            'Fribourg, Mon-Repos',
         'Fribourg, Cité-Jardins',             'Fribourg, Boschung',
   'Fribourg, Tilleul/Cathédrale',            'Fribourg, St-Pierre',
    'Fribourg/Freiburg, Pl. Gare',  'Villars-sur-Glâne, Méridienne',
      'Villars-sur-Glâne, Moncor', 'Villars-sur-Glâne, Belle-Croix',
 'Villars-sur-Glâne,Villars-Vert',             'Fribourg, Bertigny',
             'Fribourg, Bellevue',     'Fribourg, Schönberg Dunant',
             'Fribourg, Guintzet', 'Villars-sur-Glâne,Jean Paul II',
  'Villars-sur-Glâne, Hôp. cant.',       'Fribourg, Route-de-Tavel',
              'Fribourg, Kessler',            'Fribourg, Ploetscha',
               'Fribourg, Windig',      'Fribourg, Pont-Zaehringen',
           'Fribourg, Charmettes',            'Fribourg, Industrie',
              'Fribourg, J. Vogt',                'Fribourg, Fries',
              'Fribourg, Gambach',               'Fribourg, Vuille',
          'Givisiez,

In [437]:
print(travel_times_17_19['origin_name'].value_counts())
print(travel_times_17_19['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  364
Fribourg, Mon-Repos               364
Fribourg, Cité-Jardins            364
Fribourg, Boschung                364
Fribourg, Tilleul/Cathédrale      364
Fribourg, St-Pierre               364
Fribourg/Freiburg, Pl. Gare       364
Villars-sur-Glâne, Méridienne     364
Villars-sur-Glâne, Moncor         364
Villars-sur-Glâne, Belle-Croix    364
Villars-sur-Glâne,Villars-Vert    364
Fribourg, Bertigny                364
Fribourg, Bellevue                364
Fribourg, Schönberg Dunant        364
Fribourg, Guintzet                364
Villars-sur-Glâne,Jean Paul II    364
Villars-sur-Glâne, Hôp. cant.     364
Fribourg, Route-de-Tavel          364
Fribourg, Kessler                 364
Fribourg, Ploetscha               364
Fribourg, Windig                  364
Fribourg, Pont-Zaehringen         364
Fribourg, Charmettes              364
Fribourg, Industrie               364
Fribourg, J. Vogt                 364
Fribourg, Fries                   364


In [438]:
print(travel_times_17_19["dest_name"].value_counts())
print(travel_times_17_19["dest_name"].value_counts().unique())

dest_name
Fribourg, Mon-Repos               364
Fribourg, Cité-Jardins            364
Fribourg, Boschung                364
Fribourg, Tilleul/Cathédrale      364
Fribourg, St-Pierre               364
Fribourg/Freiburg, Pl. Gare       364
Villars-sur-Glâne, Méridienne     364
Villars-sur-Glâne, Moncor         364
Villars-sur-Glâne, Belle-Croix    364
Fribourg, Chaley                  364
Villars-sur-Glâne,Villars-Vert    364
Fribourg, Bertigny                364
Fribourg, Bellevue                364
Fribourg, Schönberg Dunant        364
Fribourg, Guintzet                364
Villars-sur-Glâne,Jean Paul II    364
Villars-sur-Glâne, Hôp. cant.     364
Fribourg, Route-de-Tavel          364
Fribourg, Kessler                 364
Fribourg, Ploetscha               364
Fribourg, Windig                  364
Fribourg, Pont-Zaehringen         364
Fribourg, Charmettes              364
Fribourg, Industrie               364
Fribourg, J. Vogt                 364
Fribourg, Fries                   364
Fr

In [439]:
print(travel_times_17_19.value_counts(['origin_name', 'dest_name']))
print(travel_times_17_19.value_counts(['origin_name', 'dest_name']).unique())

origin_name            dest_name                   
Fribourg, Chaley       Fribourg, Mon-Repos             13
                       Fribourg, Cité-Jardins          13
                       Fribourg, Boschung              13
                       Fribourg, Tilleul/Cathédrale    13
                       Fribourg, St-Pierre             13
                                                       ..
Givisiez, Mont Carmel  Fribourg, Industrie             13
                       Fribourg, J. Vogt               13
                       Fribourg, Fries                 13
                       Fribourg, Gambach               13
                       Fribourg, Vuille                13
Name: count, Length: 812, dtype: int64
[13]


In [440]:
o_d_example_17_19 = travel_times_17_19[
    (travel_times_17_19['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_17_19['dest_name'] == 'Fribourg, Boschung')
]
o_d_example_17_19

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 17:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 17:10:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 17:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,159,2.65,OK
2026-02-12 17:30:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 17:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 17:50:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,151,2.52,OK
2026-02-12 18:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,150,2.50,OK
2026-02-12 18:10:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,148,2.47,OK
2026-02-12 18:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,148,2.47,OK


## 1.3 Travel times 19-21:

In [441]:
travel_times_19_21 = pd.read_csv("travel_times_19_21.csv", index_col=0)

In [442]:
origins_19_21 = pd.unique(travel_times_19_21["origin_name"])
print(origins_19_21)

<StringArray>
[              'Fribourg, Chaley',            'Fribourg, Mon-Repos',
         'Fribourg, Cité-Jardins',             'Fribourg, Boschung',
   'Fribourg, Tilleul/Cathédrale',            'Fribourg, St-Pierre',
    'Fribourg/Freiburg, Pl. Gare',  'Villars-sur-Glâne, Méridienne',
      'Villars-sur-Glâne, Moncor', 'Villars-sur-Glâne, Belle-Croix',
 'Villars-sur-Glâne,Villars-Vert',             'Fribourg, Bertigny',
             'Fribourg, Bellevue',     'Fribourg, Schönberg Dunant',
             'Fribourg, Guintzet', 'Villars-sur-Glâne,Jean Paul II',
  'Villars-sur-Glâne, Hôp. cant.',       'Fribourg, Route-de-Tavel',
              'Fribourg, Kessler',            'Fribourg, Ploetscha',
               'Fribourg, Windig',      'Fribourg, Pont-Zaehringen',
           'Fribourg, Charmettes',            'Fribourg, Industrie',
              'Fribourg, J. Vogt',                'Fribourg, Fries',
              'Fribourg, Gambach',               'Fribourg, Vuille',
          'Givisiez,

In [443]:
print(travel_times_19_21['origin_name'].value_counts())
print(travel_times_19_21['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  196
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196


In [444]:
print(travel_times_19_21["dest_name"].value_counts())
print(travel_times_19_21["dest_name"].value_counts().unique())

dest_name
Fribourg, Mon-Repos               196
Fribourg, Cité-Jardins            196
Fribourg, Boschung                196
Fribourg, Tilleul/Cathédrale      196
Fribourg, St-Pierre               196
Fribourg/Freiburg, Pl. Gare       196
Villars-sur-Glâne, Méridienne     196
Villars-sur-Glâne, Moncor         196
Villars-sur-Glâne, Belle-Croix    196
Fribourg, Chaley                  196
Villars-sur-Glâne,Villars-Vert    196
Fribourg, Bertigny                196
Fribourg, Bellevue                196
Fribourg, Schönberg Dunant        196
Fribourg, Guintzet                196
Villars-sur-Glâne,Jean Paul II    196
Villars-sur-Glâne, Hôp. cant.     196
Fribourg, Route-de-Tavel          196
Fribourg, Kessler                 196
Fribourg, Ploetscha               196
Fribourg, Windig                  196
Fribourg, Pont-Zaehringen         196
Fribourg, Charmettes              196
Fribourg, Industrie               196
Fribourg, J. Vogt                 196
Fribourg, Fries                   196
Fr

In [445]:
print(travel_times_19_21.value_counts(['origin_name', 'dest_name']))
print(travel_times_19_21.value_counts(['origin_name', 'dest_name']).unique())

origin_name            dest_name                   
Fribourg, Chaley       Fribourg, Mon-Repos             7
                       Fribourg, Cité-Jardins          7
                       Fribourg, Boschung              7
                       Fribourg, Tilleul/Cathédrale    7
                       Fribourg, St-Pierre             7
                                                      ..
Givisiez, Mont Carmel  Fribourg, Industrie             7
                       Fribourg, J. Vogt               7
                       Fribourg, Fries                 7
                       Fribourg, Gambach               7
                       Fribourg, Vuille                7
Name: count, Length: 812, dtype: int64
[7]


In [446]:
o_d_example_19_21 = travel_times_19_21[
    (travel_times_19_21['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_19_21['dest_name'] == 'Fribourg, Boschung')
]
o_d_example_19_21

,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
departure_time,,,,,,,,,,,,,,
2026-02-12 19:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 19:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,154,2.57,OK
2026-02-12 19:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,144,2.40,OK
2026-02-12 20:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,151,2.52,OK
2026-02-12 20:20:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,151,2.52,OK
2026-02-12 20:40:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,153,2.55,OK
2026-02-12 21:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.3,158,2.63,OK


The csv files contain departure times between 15-17, 17-19 and 19-21 om respectively. The departures have a frequency of 20 minutes in the 15-17 and 19-21 files, whereas the 17-19 shows 10 minute intervals. For that reason, each OD pairing is present 7 times in the former and 13 times in the latter.

## 1.4 Combined travel times:

In [447]:
travel_times = pd.concat([travel_times_15_17, travel_times_17_19, travel_times_19_21], ignore_index=False)

In [448]:
travel_times.index.name = "departure_time"

In [449]:
travel_times.reset_index(inplace=True)


In [450]:
travel_times.head()

,departure_time,origin_station_id,origin_name,origin_lat,origin_lon,dest_station_id,dest_name,dest_lat,dest_lon,distance_meters,duration_seconds,duration_minutes,duration_in_traffic_seconds,duration_in_traffic_minutes,status
0,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589152,"Fribourg, Mon-Repos",46.806711,7.172136,270,35,0.58,42,0.70,OK
1,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589138,"Fribourg, Cité-Jardins",46.809385,7.170446,659,86,1.43,117,1.95,OK
2,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8591766,"Fribourg, Boschung",46.811451,7.171016,1013,138,2.30,174,2.90,OK
3,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8587255,"Fribourg, Tilleul/Cathédrale",46.806090,7.161261,3788,445,7.42,506,8.43,OK
4,2026-02-12 15:00:00,8589141,"Fribourg, Chaley",46.806281,7.175601,8589161,"Fribourg, St-Pierre",46.803911,7.155266,4335,561,9.35,622,10.37,OK


In [451]:
print(travel_times['origin_name'].value_counts())
print(travel_times['origin_name'].value_counts().unique())

origin_name
Fribourg, Chaley                  756
Fribourg, Mon-Repos               756
Fribourg, Cité-Jardins            756
Fribourg, Boschung                756
Fribourg, Tilleul/Cathédrale      756
Fribourg, St-Pierre               756
Fribourg/Freiburg, Pl. Gare       756
Villars-sur-Glâne, Méridienne     756
Villars-sur-Glâne, Moncor         756
Villars-sur-Glâne, Belle-Croix    756
Villars-sur-Glâne,Villars-Vert    756
Fribourg, Bertigny                756
Fribourg, Bellevue                756
Fribourg, Schönberg Dunant        756
Fribourg, Guintzet                756
Villars-sur-Glâne,Jean Paul II    756
Villars-sur-Glâne, Hôp. cant.     756
Fribourg, Route-de-Tavel          756
Fribourg, Kessler                 756
Fribourg, Ploetscha               756
Fribourg, Windig                  756
Fribourg, Pont-Zaehringen         756
Fribourg, Charmettes              756
Fribourg, Industrie               756
Fribourg, J. Vogt                 756
Fribourg, Fries                   756


In [452]:
travel_times['departure_time'] = pd.to_datetime(travel_times['departure_time'])

In [453]:
travel_times.dtypes["departure_time"]

dtype('<M8[us]')

# 2. Model Preparation

$$\min\sum_{k\in K}\sum_{i\in N}\sum_{j\in N}b_1 c^kt_{ij}x_{ij}^k+\sum_{i\in P}b_2c^0(1-\sum_{k\in K}\sum_{j\in P\cup D}x_{ij}^k)+b_3E[Q(x,w,a,q,\xi)]$$


## 2.1 Setting parameters for model

In [454]:
# Base parameters
mini_bus_number = 6
minibus_capacities = [4, 6, 8]
travel_cost_mini_bus = [40, 30, 20]
num_buses_per_category = [2, 2, 2]
time_interval = 5
c_0 = 100

# Calibration parameters
b_1 = 1.0
b_2 = 1.0
b_3 = 1.0

# Time
time_of_interest = '2026-02-12 15:00:00'

## 2.1 Extracting the node indices

In [455]:
nodes = pd.unique(travel_times["origin_station_id"])
nodes_idx_map = {node: i for i, node in enumerate(nodes)}
nodes_idx = list(nodes_idx_map.values())

In [456]:
nodes_idx_map

{np.int64(8589141): 0,
 np.int64(8589152): 1,
 np.int64(8589138): 2,
 np.int64(8591766): 3,
 np.int64(8587255): 4,
 np.int64(8589161): 5,
 np.int64(8592374): 6,
 np.int64(8589130): 7,
 np.int64(8589131): 8,
 np.int64(8588344): 9,
 np.int64(8577786): 10,
 np.int64(8577785): 11,
 np.int64(8504622): 12,
 np.int64(8589271): 13,
 np.int64(8592375): 14,
 np.int64(8592378): 15,
 np.int64(8592377): 16,
 np.int64(8591767): 17,
 np.int64(8589270): 18,
 np.int64(8589147): 19,
 np.int64(8589158): 20,
 np.int64(8587356): 21,
 np.int64(8577741): 22,
 np.int64(8589154): 23,
 np.int64(8588858): 24,
 np.int64(8589155): 25,
 np.int64(8588351): 26,
 np.int64(8577820): 27,
 np.int64(8587238): 28}

## 2.2 Extracting the travel times

In [457]:
travel_times_filtered = travel_times[travel_times['departure_time'] == time_of_interest]
travel_times_filtered['origin_idx'] = travel_times_filtered['origin_station_id'].map(nodes_idx_map)
travel_times_filtered['dest_idx'] = travel_times_filtered['dest_station_id'].map(nodes_idx_map)


In [458]:
od_travel_time = travel_times_filtered.pivot(index='origin_idx', 
                                  columns='dest_idx', 
                                  values='duration_in_traffic_minutes').fillna(0)

# Show first few rows
od_travel_time.head()

dest_idx,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
origin_idx,,,,,,,,,,,,,,,,,,,,,
0,0.00,0.70,1.95,2.90,8.43,10.37,12.48,14.43,12.57,12.62,...,3.28,4.18,3.00,14.37,15.08,14.93,15.48,10.55,8.92,11.25
1,0.75,0.00,1.22,2.30,7.70,9.60,11.77,13.73,11.88,12.07,...,2.65,3.50,2.37,13.65,14.38,14.23,14.77,9.85,8.22,10.52
2,1.68,0.92,0.00,1.05,6.68,8.52,10.70,12.70,10.87,10.98,...,1.53,2.52,2.70,13.92,14.72,13.33,13.75,8.82,7.15,9.52
3,2.70,1.78,0.85,0.00,5.65,7.50,9.73,11.73,9.87,10.03,...,1.07,1.93,1.58,13.63,14.38,12.33,12.72,7.80,6.13,8.42
4,8.30,7.60,6.68,5.77,0.00,2.13,4.25,13.37,11.25,11.42,...,6.70,7.60,6.62,9.28,10.13,7.95,8.45,4.48,5.32,7.48


In [459]:
# Filter the row for the specific origin and destination
od_pair = travel_times_filtered[
    (travel_times_filtered['origin_name'] == 'Fribourg, Chaley') &
    (travel_times_filtered['dest_name'] == 'Fribourg, Tilleul/Cathédrale')
]

# Include the indices in the output
print(od_pair[['origin_name', 'origin_idx', 'dest_name', 'dest_idx', 'duration_in_traffic_minutes']])

        origin_name  origin_idx                     dest_name  dest_idx  \
3  Fribourg, Chaley           0  Fribourg, Tilleul/Cathédrale         4   

   duration_in_traffic_minutes  
3                         8.43  


In [460]:
# Access the specific OD pair
travel_time = od_travel_time.loc[0, 4]
print(travel_time)

8.43


## 2.3 Minimus indices

In [461]:
bus_idx = list(range(0, mini_bus_number))
bus_idx

[0, 1, 2, 3, 4, 5]

In [462]:
bus_cost = {}
current_bus = 0
for cost, count in zip(travel_cost_mini_bus, num_buses_per_category):
    for _ in range(count):
        bus_cost[current_bus] = cost
        current_bus += 1

bus_capacity = {}
for capacity, count in zip(minibus_capacities, num_buses_per_category):
    for _ in range(count):
        bus_capacity[current_bus] = capacity
        current_bus += 1

In [463]:
bus_cost

{0: 40, 1: 40, 2: 30, 3: 30, 4: 20, 5: 20}

In [464]:
bus_capacity

{6: 4, 7: 4, 8: 6, 9: 6, 10: 8, 11: 8}

## 2.4 Model definition

We'll consider a simple model to construct the MILP formulation and then extend it onto the complete network. The following network will be considered at time t:

<center><img src="Simple_Network.jpeg" width="400" /></center>

In [465]:
#N_Base = [[0, 1, 2, 3, 4]]
#P = [] # set of pickup nodes
#D = [] # set of dropoff nodes
#S = [] # set of vehicle initial position nodes
#Z = [] # set of vehicle virtual destination nodes
#u = [] # Matching relationships
#q = [] # Number of passengers in request
#q_k = [] # Number of passengers in bus k
#M = 1000 # Large number
#w = 1 # Wait time + service time at node i
#a = [] # arrival time at nodes
#w_max = 30 # Maximum waiting time of a vehicle at a node
#tep = [] # Earliest pickup time
#tlp = [] # Latest pickup time
#tld = [] # Latest dropoff time
#mu = [] # Node position in tour

### 2.4.0 Constants

In [466]:
N_Base = [[0, 1, 2, 3, 4]]
M = 1000 # Large number
w = 1 # Wait time + service time at node i
w_max = 30 # Maximum waiting time of a vehicle at a node
b_1 = 1
b_2 = 1
b_3 = 1
c_0 = 1000
a_max = 2 # Maximum detour tolerated by passenger (2 * travel time)
pax_max_wait = 10 # We assume a 10 minute window for pickup (tlp - tep)

### 2.4.1 Set of requests:

A request $r\in R, r=\{1,\ldots,|R|\}$ is represented as a vector: $(q_r, p_r, d_r)$, $q_r$ is the number of passengers in this request, $p_r$ is the pickup node, $d_r$ the drop-off node and $|R|$ is the number of requests considered in the horizon. There are three types of requests:
 - $r\in R_{waiting}$: waiting requests
 - $r\in R_{scheduled}$: scheduled requests (those that were assigned to certain vehicles; passengers could either be waiting or already on the bus)
 - $r\in R_{predicted}$: predicted requests in the future

To build the initial model we'll consider the following requests, which are all in the waiting pool:

In [467]:
# Requests considered (passengers, origin, destination)
r_1 = [2, 1, 5]
r_2 = [4, 2, 4]
r_3 = [1, 1, 4]
r_4 = [3, 2, 5]
r_5 = [2, 1, 4]
r_6 = [1, 5, 3]
r_7 = [4, 4, 2]
r_8 = [2, 3, 1]
r_9 = [3, 3, 4]
r_10 = [1, 5, 1]
R_wait = [r_1, r_2, r_3, r_4, r_5, r_6, r_7, r_8, r_9, r_10]
R_sched = []
R_pred = []
R = R_wait + R_sched

n_req = len(R)

In [468]:
print(R)
print(n_req)

[[2, 1, 5], [4, 2, 4], [1, 1, 4], [3, 2, 5], [2, 1, 4], [1, 5, 3], [4, 4, 2], [2, 3, 1], [3, 3, 4], [1, 5, 1]]
10


### 2.4.2 Vehicle attributes:
$K$ is the set of vehicles available during each time step, each vehicle $k\in K, k=\{1,\ldots,|K|\}$ is represented as a vector: $(q_{max}^k,V_0^k,V_d^k)$. In this vector $q_{max}^k$ represents the number of remaining seats, $V_0^k$ the current position, $V_d^k$ the virtual destination and $|K|$ the number of vehicles. The vehicle returns to the virtual destination after an entire day of operation, the distance from all other locations to the virtual destination is set to 0. $c_k$ is the unit travel cost and $c^0$ the penalty for rejecting a request.

In [469]:
# Vehicles (maximum capacity, current position, virtual destination)
k_1 = [8, 3, 0]
k_2 = [6, 5, 0]
K = [k_1, k_2]
bus_idx = list(range(0, len(K)))

bus_cost = {0: 40, 1: 30}

In [470]:
print(K)
print(bus_idx)
print(bus_cost)

[[8, 3, 0], [6, 5, 0]]
[0, 1]
{0: 40, 1: 30}


### 2.4.3 Node set:
$N=P\cup D\cup S\cup Z$ is defined as the node set and contains all the pickup nodes $P$, dropoff nodes $D$, $S$ vehicle initial positions, $Z$ set of virtual destination nodes.

More specifically, $P$ is composed of of three subsets, the nodes of waiting $P_{wait}$, the nodes of scheduled $P_{sched}$ and predicted passengers $P_{pred}$. $P$ is the union of these subsets $P=P_{wait}\cup P_{sched}\cup P_{pred}$.

In [471]:
#P_wait = [R_wait[1] for R_wait in R_wait]
#P_sched = [R_sched[1] for R_sched in R_sched]
#P_pred = [R_pred[1] for R_pred in R_pred]

#P = P_wait + P_sched
#P

In [472]:
#D_wait = [R_wait[2] for R_wait in R_wait]
#D_sched = [R_sched[2] for R_sched in R_sched]
#D_pred = [R_pred[2] for R_pred in R_pred]

#D = D_wait + D_sched
#D

In [473]:
#P_and_D = P + D
#P_and_D

In [474]:
#S = [K[1] for K in K]
#S

In [475]:
#Z = [K[2] for K in K]
#Z

In [476]:
#N = P_and_D + S + Z
#N

In [477]:
#s = np.array([1, 1, 1, 1, 1])

#t = np.array([[0, 2, 3, 2, 4],
#     [3, 0, 4, 7, 2],
#     [2, 3, 0, 3, 4],
#     [2, 6, 1, 0, 3],
#     [5, 4, 5, 2, 0]])

# Your logical node numbers
#nodes_logical = [1,2,3,4,5]

# Map logical numbers to 0-based matrix indices
#node_to_idx = {1:0, 2:1, 3:2, 4:3, 5:4}

# Create dictionary travel times keyed by logical nodes
#t_dict = {(i,j): t[node_to_idx[i], node_to_idx[j]] for i in nodes_logical for j in nodes_logical}

#s_dict = {i: s[node_to_idx[i]] for i in nodes_logical}

#w_max = 10

In [478]:
# Creating unique node indices for each requested pickup (also separated into wait and sched)
P_nodes = list(range(n_req))
P_wait = list(range(len(R_wait)))
P_sched = list(range(len(R_wait), len(R_wait) + len(R_sched)))

# Creating unique node indices for each requested dropoff
D_nodes = list(range(n_req, 2*n_req))

# Creating unique node indices for each vehicle initial position and virtual destination
S_nodes = list(range(2*n_req, 2*n_req + len(K)))
Z_nodes = list(range(2*n_req + len(K), 2*n_req + 2*len(K)))

# Creating node sets
P_and_D = P_nodes + D_nodes
N = P_nodes + D_nodes + S_nodes + Z_nodes

In [479]:
print(P_and_D)
print(N)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


In [480]:
# Creating mapping from modeling nodes to physical nodes
P_loc = {i: R[i][1] for i in range(n_req)}
D_loc = {i+n_req: R[i][2] for i in range(n_req)}
S_loc = {S_nodes[k]: K[k][1] for k in range(len(K))}
Z_loc = {Z_nodes[k]: K[k][2] for k in range(len(K))}

node_to_loc = {}
node_to_loc.update(P_loc)
node_to_loc.update(D_loc)
node_to_loc.update(S_loc)
node_to_loc.update(Z_loc)

In [481]:
print(node_to_loc)

{0: 1, 1: 2, 2: 1, 3: 2, 4: 1, 5: 5, 6: 4, 7: 3, 8: 3, 9: 5, 10: 5, 11: 4, 12: 4, 13: 5, 14: 4, 15: 3, 16: 2, 17: 1, 18: 4, 19: 1, 20: 3, 21: 5, 22: 0, 23: 0}


### 2.4.4 Travel and service times:

$s_i$ denotes the service time of a pickup or dropoff node $i$, $i\in P\cup D$. $t_{ij}$ is the travel time between two notes, $w_{max}$ the maximum waiting time of a vehicle at a station. In this example we'll consider that the service time is constant for all the nodes and is approximately 1 minute, the travel times were defined randomly and the maximum waiting time was considered to be 10 minutes at a node.

In [482]:
s = np.array([0, 1, 1, 1, 1, 1])

t = np.array([[0, 0, 0, 0, 0, 0],
     [0, 0, 2, 3, 2, 4],
     [0, 3, 0, 4, 7, 2],
     [0, 2, 3, 0, 3, 4],
     [0, 2, 6, 1, 0, 3],
     [0, 5, 4, 5, 2, 0]])

node_to_idx = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5}

# Create dictionary travel times keyed by logical nodes
t_dict = {(i,j): t[node_to_idx[node_to_loc[i]], node_to_idx[node_to_loc[j]]] for i in N for j in N}

s_dict = {i: s[node_to_idx[node_to_loc[i]]] for i in N}

For a given node $i\in P$, the time window is $[tep_i, tlp_i]$, where $tep_i$ represents the passenger's earliest pickup time and $[tlp_i]$ the latest pickup time. Similarly, $[ted_i, tld_i]$, where $ted_i$ represent the passenger's earliest dropoff time and $[tld_i]$ the latest dropoff time. $ted_i$ was set to zero if there are no constraints on the earliest dropoff time. $a_{max}\geq 1$ represents the passenger's tolerance for increased travel time compared with the shortest travel time. We'll assume for the sake of modeling that separate arrays following the same order as the requests provides the earliest pick-up times and that the other values are derived using detour tolerance and waiting time at a station.

In [483]:
req_t_p_wait = [10, 25, 15, 40, 30, 60, 20, 50, 45, 10] 
req_t_p_sched = [] 
req_t_p_pred = [] 

req_t_p = req_t_p_wait + req_t_p_sched

tep = {} 
tlp = {} 
ted = {} 
tld = {} 

for idx, i in enumerate(P_nodes):
    tep[i] = req_t_p[idx]
    tlp[i] = tep[i] + pax_max_wait

for idx, i in enumerate(D_nodes):
    pickup_node = i - n_req 
    travel_time = t_dict[pickup_node, i]
    ted[i] = 0 # We'll assume no constraint on earliest drop-off time in a first step
    tld[i] = tep[pickup_node] + (travel_time * a_max)

e_dict = {}
l_dict = {}

for i in P_nodes:
    e_dict[i] = tep[i]
    l_dict[i] = tlp[i]

for i in D_nodes:
    e_dict[i] = ted[i]
    l_dict[i] = tld[i]

for i in S_nodes + Z_nodes:
    e_dict[i] = 0
    l_dict[i] = 1440 # 1 full day in minutes

### 2.4.5 Capacity

In [484]:
#Q = [R[0] for R in R_wait] + [R[0] for R in R_sched] + [-R[0] for R in R_wait] + [-R[0] for R in R_sched] + [0 for K in K] + [0 for K in K]

#Q

Q = {}

for i in P_nodes:
    Q[i] = R[i][0]

for i in D_nodes:
    Q[i] = -R[i-n_req][0]

for i in S_nodes + Z_nodes:
    Q[i] = 0
    
Q_max = [K[0] for K in K]

ub_dict = {(i, k): Q_max[k] for i in N for k in bus_idx}

In [485]:
print(Q)
print(Q_max)
print(ub_dict)

{0: 2, 1: 4, 2: 1, 3: 3, 4: 2, 5: 1, 6: 4, 7: 2, 8: 3, 9: 1, 10: -2, 11: -4, 12: -1, 13: -3, 14: -2, 15: -1, 16: -4, 17: -2, 18: -3, 19: -1, 20: 0, 21: 0, 22: 0, 23: 0}
[8, 6]
{(0, 0): 8, (0, 1): 6, (1, 0): 8, (1, 1): 6, (2, 0): 8, (2, 1): 6, (3, 0): 8, (3, 1): 6, (4, 0): 8, (4, 1): 6, (5, 0): 8, (5, 1): 6, (6, 0): 8, (6, 1): 6, (7, 0): 8, (7, 1): 6, (8, 0): 8, (8, 1): 6, (9, 0): 8, (9, 1): 6, (10, 0): 8, (10, 1): 6, (11, 0): 8, (11, 1): 6, (12, 0): 8, (12, 1): 6, (13, 0): 8, (13, 1): 6, (14, 0): 8, (14, 1): 6, (15, 0): 8, (15, 1): 6, (16, 0): 8, (16, 1): 6, (17, 0): 8, (17, 1): 6, (18, 0): 8, (18, 1): 6, (19, 0): 8, (19, 1): 6, (20, 0): 8, (20, 1): 6, (21, 0): 8, (21, 1): 6, (22, 0): 8, (22, 1): 6, (23, 0): 8, (23, 1): 6}


### 2.4.6 Matching relationship:
$u_i^k$ represents the matching relationship between vehicle $k$ and node $i$ ensuring that the scheduled request is served by a predetermined vehicle and is defined for $r\in R_{scheduled}$ only. If vehicle $k$ passes through node $i$, then $u_i^k=1$ and $0$ otherwise. To start of this model, we'll assume no matches were made yet.

In [ ]:
# Your logical node numbers
nodes_logical = [0,1,2,3,4,5]

u = np.zeros((len(bus_idx), len(nodes_logical)))

# Map logical numbers to 0-based matrix indices
node_to_idx = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5}

# Create dictionary travel times keyed by logical nodes
u_dict = {(k, i): u[k, node_to_idx[node_to_loc[i]]] for i in N for k in bus_idx}

In [487]:
#u

The matrix $A$ is composed of elements $A_i^m$, which take value 1 if the physical station corresponds to node $i$ and 0 otherwise for all $i\in P\cup D$.

In [488]:
M_stations = list(set(node_to_loc.values()))

In [489]:
print(M_stations)

[0, 1, 2, 3, 4, 5]


In [490]:
node_to_loc

{0: 1,
 1: 2,
 2: 1,
 3: 2,
 4: 1,
 5: 5,
 6: 4,
 7: 3,
 8: 3,
 9: 5,
 10: 5,
 11: 4,
 12: 4,
 13: 5,
 14: 4,
 15: 3,
 16: 2,
 17: 1,
 18: 4,
 19: 1,
 20: 3,
 21: 5,
 22: 0,
 23: 0}

In [491]:
A_m = {}

# i is the logical node, m is the physical station
for i in P_and_D:
    for m in M_stations:
        if node_to_loc[i] == m:
            A_m[i, m] = 1
        else:
            A_m[i, m] = 0

In [492]:
A_m

{(0, 0): 0,
 (0, 1): 1,
 (0, 2): 0,
 (0, 3): 0,
 (0, 4): 0,
 (0, 5): 0,
 (1, 0): 0,
 (1, 1): 0,
 (1, 2): 1,
 (1, 3): 0,
 (1, 4): 0,
 (1, 5): 0,
 (2, 0): 0,
 (2, 1): 1,
 (2, 2): 0,
 (2, 3): 0,
 (2, 4): 0,
 (2, 5): 0,
 (3, 0): 0,
 (3, 1): 0,
 (3, 2): 1,
 (3, 3): 0,
 (3, 4): 0,
 (3, 5): 0,
 (4, 0): 0,
 (4, 1): 1,
 (4, 2): 0,
 (4, 3): 0,
 (4, 4): 0,
 (4, 5): 0,
 (5, 0): 0,
 (5, 1): 0,
 (5, 2): 0,
 (5, 3): 0,
 (5, 4): 0,
 (5, 5): 1,
 (6, 0): 0,
 (6, 1): 0,
 (6, 2): 0,
 (6, 3): 0,
 (6, 4): 1,
 (6, 5): 0,
 (7, 0): 0,
 (7, 1): 0,
 (7, 2): 0,
 (7, 3): 1,
 (7, 4): 0,
 (7, 5): 0,
 (8, 0): 0,
 (8, 1): 0,
 (8, 2): 0,
 (8, 3): 1,
 (8, 4): 0,
 (8, 5): 0,
 (9, 0): 0,
 (9, 1): 0,
 (9, 2): 0,
 (9, 3): 0,
 (9, 4): 0,
 (9, 5): 1,
 (10, 0): 0,
 (10, 1): 0,
 (10, 2): 0,
 (10, 3): 0,
 (10, 4): 0,
 (10, 5): 1,
 (11, 0): 0,
 (11, 1): 0,
 (11, 2): 0,
 (11, 3): 0,
 (11, 4): 1,
 (11, 5): 0,
 (12, 0): 0,
 (12, 1): 0,
 (12, 2): 0,
 (12, 3): 0,
 (12, 4): 1,
 (12, 5): 0,
 (13, 0): 0,
 (13, 1): 0,
 (13, 2): 0,
 (13, 3

### 2.4.7 Model definition:

In [493]:
# Initializing the model
model_MILP_base = gp.Model("MILP_base")
model_MILP_base.Params.OutputFlag = 1

# Initializing the decision variables
x_base = model_MILP_base.addVars(N, N, bus_idx, vtype=gp.GRB.BINARY, name="x")
q_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.INTEGER, lb=0, ub=ub_dict, name="q_k")
w_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=0, ub=w_max, name="w_k")
a_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, name="a_k")
mu = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=1, ub=len(N), name="mu")
y = model_MILP_base.addVars(P_nodes, vtype=gp.GRB.BINARY, name="y")

# Objective funtion to be minimized
obj_expr_trav_cost = gp.quicksum(
    b_1 * bus_cost[k] * t_dict[i, j] * x_base[i, j, k]
    for i in N
    for j in N
    for k in bus_idx
)

for i in P_nodes:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[i, j, k] for j in P_and_D for k in bus_idx) == y[i]
    )

obj_expr_reject_cost = gp.quicksum(
    b_2 * c_0 * (1 - y[i]) for i in P_nodes
)

# obj_expr_recourse_cost = b_3 * E

model_MILP_base.setObjective(obj_expr_trav_cost + obj_expr_reject_cost, gp.GRB.MINIMIZE)

# Constraints

# Previous scheduled request served
for i in P_sched:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) == u_dict[k, i]
        )

# Each request served at most once
for i in P_wait:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[i, j, k] for j in N for k in bus_idx) <= 1
    )

# Ensures same vehicle visits pickup and drop-off nodes of same request
for i in P_nodes:
    d = i + n_req
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[d, j, k] for j in N) == 0
        )

# Flow conservation constraints
for i in P_nodes + D_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[j, i, k] for j in N) == 0
        )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[S_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 1
    )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[Z_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, Z_nodes[k], k] for j in N) == -1
    )

# Capacity constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                q_k[i, k] + Q[i] - M * (1 - x_base[i, j, k]) <= q_k[j, k]
            )
            model_MILP_base.addConstr(
                q_k[j, k] <= q_k[i, k] + Q[i] + M * (1 - x_base[i, j, k])
            )

#for i_idx, i in enumerate(N):
#    for k in bus_idx:
#        model_MILP_base.addConstr(
#            q_k[i_idx, k] <= Q_max[k]
#        ) ######## Upper bound constraint is already included in decision variable definition

# Time constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, j] - M * (1 - x_base[i, j, k]) <= a_k[j, k]
            )

for m in M_stations:
    model_MILP_base.addConstr(
        gp.quicksum(A_m[i, m] * w_k[i, k] for i in P_and_D for k in bus_idx) <= w_max
    )

for i in P_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            e_dict[i] <= a_k[i, k] + w_k[i, k] + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
        )
        model_MILP_base.addConstr(
            a_k[i, k] <= l_dict[i] + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
        )

for i in D_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i, k] <= l_dict[i] + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
        )

for i in P_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i+n_req, k] - a_k[i, k] <= a_max * t_dict[i, i+n_req] 
            + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
        )

for i in P_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, i+n_req] <= a_k[i+n_req, k] 
            + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
        )

for i in N:
    for j in N:
        for k in bus_idx:
            if i != j and i in P_nodes + D_nodes and j in P_nodes + D_nodes:
                model_MILP_base.addConstr(
                    mu[i, k] - mu[j, k] + M * x_base[i, j, k] <= M - 1,
                    name=f"subtour_{i}_{j}_{k}"
                )

for i in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                x_base[i, i, k] == 0
            )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 0
    ) # Vehicle cannot enter start node

for k in bus_idx: # vehicle cannot enter or leave any other vehicle's start or end node
    for k_other in bus_idx:
        if k != k_other:
            # forbid vehicle k from visiting start of other vehicles
            model_MILP_base.addConstr(
                gp.quicksum(x_base[i, S_nodes[k_other], k] for i in N) == 0
            )
            model_MILP_base.addConstr(
                gp.quicksum(x_base[S_nodes[k_other], j, k] for j in N) == 0
            )

            # forbid vehicle k from visiting end of other vehicles
            model_MILP_base.addConstr(
                gp.quicksum(x_base[i, Z_nodes[k_other], k] for i in N) == 0
            )
            model_MILP_base.addConstr(
                gp.quicksum(x_base[Z_nodes[k_other], j, k] for j in N) == 0
            )

Set parameter OutputFlag to value 1


In [494]:
model_MILP_base.optimize()

if model_MILP_base.status == gp.GRB.OPTIMAL:
    print("Optimal solution found")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4464 rows, 1354 columns and 20294 nonzeros (Min)
Model fingerprint: 0xd868fd35
Model has 778 linear objective coefficients and an objective constant of 10000
Variable types: 144 continuous, 1210 integer (1162 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [3e+01, 1e+03]
  Bounds range     [1e+00, 3e+01]
  RHS range        [1e+00, 1e+03]

Found heuristic solution: objective 10000.000000
Presolve removed 1001 rows and 372 columns
Presolve time: 0.07s
Presolved: 3463 rows, 982 columns, 29965 nonzeros
Variable types: 104 continuous, 878 integer (834 binary)

Root relaxation: objective -1.818989e-12, 150 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf 

In [495]:
active_arcs = [(i, j, k) for i in N for j in N for k in bus_idx
               if x_base[i, j, k].X > 0.5]

routes = {}

for k in bus_idx:
    # Start from vehicle start node
    current = S_nodes[k]
    route = [current]
    
    while True:
        next_nodes = [j for (i, j, kk) in active_arcs if i == current and kk == k]
        
        if not next_nodes:
            break
        
        current = next_nodes[0]
        route.append(current)
        
        if current == Z_nodes[k]:
            break
    
    routes[k] = route

for k in bus_idx:
    print(f"\nVehicle {k} route:")
    for node in routes[k]:
        print(f"Node {node} (location {node_to_loc[node]})")


Vehicle 0 route:
Node 20 (location 3)
Node 2 (location 1)
Node 12 (location 4)
Node 6 (location 4)
Node 16 (location 2)
Node 3 (location 2)
Node 13 (location 5)
Node 5 (location 5)
Node 15 (location 3)
Node 22 (location 0)

Vehicle 1 route:
Node 21 (location 5)
Node 9 (location 5)
Node 19 (location 1)
Node 1 (location 2)
Node 4 (location 1)
Node 11 (location 4)
Node 14 (location 4)
Node 8 (location 3)
Node 18 (location 4)
Node 7 (location 3)
Node 17 (location 1)
Node 23 (location 0)


In [496]:
arrival_times = {(i, k): a_k[i, k].X for i in N for k in bus_idx}

for k in bus_idx:
    print(f"\nVehicle {k} schedule:")
    for node in routes[k]:
        print(f"Node {node}: arrival = {arrival_times[node, k]:.2f}")


Vehicle 0 schedule:
Node 20: arrival = 0.00
Node 2: arrival = 15.00
Node 12: arrival = 18.00
Node 6: arrival = 19.00
Node 16: arrival = 27.00
Node 3: arrival = 40.00
Node 13: arrival = 43.00
Node 5: arrival = 60.00
Node 15: arrival = 70.00
Node 22: arrival = 1000.00

Vehicle 1 schedule:
Node 21: arrival = 13.00
Node 9: arrival = 14.00
Node 19: arrival = 20.00
Node 1: arrival = 25.00
Node 4: arrival = 30.00
Node 11: arrival = 33.00
Node 14: arrival = 34.00
Node 8: arrival = 45.00
Node 18: arrival = 49.00
Node 7: arrival = 51.00
Node 17: arrival = 54.00
Node 23: arrival = 1013.00


In [497]:
loads = {(i, k): q_k[i, k].X for i in N for k in bus_idx}

for k in bus_idx:
    print(f"\nVehicle {k} loads:")
    for node in routes[k]:
        print(f"Node {node}: load = {loads[node, k]:.0f}")


Vehicle 0 loads:
Node 20: load = -0
Node 2: load = -0
Node 12: load = 1
Node 6: load = 0
Node 16: load = 4
Node 3: load = -0
Node 13: load = 3
Node 5: load = -0
Node 15: load = 1
Node 22: load = -0

Vehicle 1 loads:
Node 21: load = -0
Node 9: load = -0
Node 19: load = 1
Node 1: load = 0
Node 4: load = 4
Node 11: load = 6
Node 14: load = 2
Node 8: load = -0
Node 18: load = 3
Node 7: load = 0
Node 17: load = 2
Node 23: load = -0


In [498]:
waiting = {(i, k): w_k[i, k].X for i in N for k in bus_idx}

served_requests = []

for i in P_nodes:
    if sum(x_base[i, j, k].X for j in N for k in bus_idx) > 0.5:
        served_requests.append(i)

print("Served requests:", served_requests)

Served requests: [1, 2, 3, 4, 5, 6, 7, 8, 9]


In [499]:
print("Objective value:", model_MILP_base.ObjVal)

Objective value: 2250.0


In [500]:
served = [i for i in P_nodes if y[i].X > 0.5]
print("Served:", served)
print("Rejected:", [i for i in P_nodes if y[i].X < 0.5])

Served: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Rejected: [0]


### 2.4.8 Model iterations

In this section we'll attempt to make the model iterate over multiple timesteps, which generate new requests and reroutes the bus to optmize over a period of time. We'll set the starting time to 0.

In [58]:
# Simulation start and end times, and interval
t_start = 0
t_end = 720
interval = 15
time_stamps = range(t_start, t_end + 1, interval)

# Constants
N_Base = [[0, 1, 2, 3, 4]]
M = 10000000 # Large number
w = 1 # Wait time + service time at node i
w_max = 30 # Maximum waiting time of a vehicle at a node
b_1 = 1
b_2 = 1
b_3 = 1
c_0 = 1000
a_max = 2 # Maximum detour tolerated by passenger (2 * travel time)
pax_max_wait = 10 # We assume a 10 minute window for pickup (tlp - tep)

In [59]:
# Initial requests (passengers, origin, destination)
r_1 = [2, 1, 5]
r_2 = [4, 2, 4]
r_3 = [1, 1, 4]
r_4 = [3, 2, 5]
r_5 = [2, 1, 4]
r_6 = [1, 5, 3]
r_7 = [4, 4, 2]
r_8 = [2, 3, 1]
r_9 = [3, 3, 4]
r_10 = [1, 5, 1]
R_wait = [r_1, r_2, r_3, r_4, r_5, r_6, r_7, r_8, r_9, r_10]
R_sched = []
R_pred = []
R = R_wait + R_sched

n_req = len(R)

# Vehicles (maximum capacity, current position, virtual destination intially)
k_1 = [8, 3, 0]
k_2 = [6, 5, 0]
K = [k_1, k_2]
bus_idx = list(range(0, len(K)))

bus_cost = {0: 40, 1: 30}

req_t_p_wait = [10, 25, 15, 40, 30, 60, 20, 50, 45, 10] 
req_t_p_sched = [] 
req_t_p_pred = [] 

req_t_p = req_t_p_wait + req_t_p_sched

In [60]:
s = np.array([0, 1, 1, 1, 1, 1])

t_matrix = np.array([[0, 0, 0, 0, 0, 0],
     [0, 0, 2, 3, 2, 4],
     [0, 3, 0, 4, 7, 2],
     [0, 2, 3, 0, 3, 4],
     [0, 2, 6, 1, 0, 3],
     [0, 5, 4, 5, 2, 0]])

node_to_idx = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5}

# Your logical node numbers
nodes_logical = [0,1,2,3,4,5]

u = {}

# Map logical numbers to 0-based matrix indices
node_to_idx = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5}

# Create dictionary travel times keyed by logical nodes
#u_dict = {(k, i): u[k, node_to_idx[node_to_loc[i]]] for i in N for k in bus_idx}



In [61]:
history_routes = {k: [] for k in bus_idx}
history_stats = {
    'time_step': [],
    'new_reqs_presented': [],
    'reqs_rejected': [],
    'obj_cost': [],
    'in_transit_carried_over': []
}

In [62]:
for t in time_stamps:
    print(f"\nStarting iteration for time {t} minutes")

    # Step 1: Creating unique node indices for each requested pickup (also separated into wait and sched)
    P_nodes = list(range(n_req))
    P_wait = list(range(len(R_wait)))
    P_sched = list(range(len(R_wait), len(R_wait) + len(R_sched)))

    # Step 2: Creating unique node indices for each requested dropoff
    D_nodes = list(range(n_req, 2*n_req))

    # Step 3: Creating unique node indices for each vehicle initial position and virtual destination
    S_nodes = list(range(2*n_req, 2*n_req + len(K)))
    Z_nodes = list(range(2*n_req + len(K), 2*n_req + 2*len(K)))

    # Step 4: Creating node sets
    P_and_D = P_nodes + D_nodes
    N = P_nodes + D_nodes + S_nodes + Z_nodes

    # Step 5: Creating mapping from modeling nodes to physical nodes
    P_loc = {i: R[i][1] for i in range(n_req)}
    D_loc = {i+n_req: R[i][2] for i in range(n_req)}
    S_loc = {S_nodes[k]: K[k][1] for k in range(len(K))}
    Z_loc = {Z_nodes[k]: K[k][2] for k in range(len(K))}

    node_to_loc = {}
    node_to_loc.update(P_loc)
    node_to_loc.update(D_loc)
    node_to_loc.update(S_loc)
    node_to_loc.update(Z_loc)

    # Create dictionary travel times keyed by logical nodes
    t_dict = {(i,j): t_matrix[node_to_idx[node_to_loc[i]], node_to_idx[node_to_loc[j]]] for i in N for j in N}

    s_dict = {i: s[node_to_idx[node_to_loc[i]]] for i in N}

    # Remove ghost boarding times for passengers already on the bus
    for idx, i in enumerate(P_sched):
        if req_t_p_sched[idx] <= t: # This identifies in-transit passengers
            s_dict[i] = 0

    tep = {} 
    tlp = {} 
    ted = {} 
    tld = {} 

    for idx, i in enumerate(P_nodes):
        tep[i] = req_t_p[idx]
        tlp[i] = tep[i] + pax_max_wait

    for idx, i in enumerate(D_nodes):
        pickup_node = i - n_req 
        travel_time = t_dict[pickup_node, i]
        ted[i] = 0 # We'll assume no constraint on earliest drop-off time in a first step
        tld[i] = tep[pickup_node] + max(travel_time * a_max, pax_max_wait + s_dict[pickup_node])

    e_dict = {}
    l_dict = {}

    for i in P_nodes:
        e_dict[i] = tep[i]
        l_dict[i] = tlp[i]

    for i in D_nodes:
        e_dict[i] = ted[i]
        l_dict[i] = tld[i]

    for i in S_nodes + Z_nodes:
        e_dict[i] = 0
        l_dict[i] = 1440 # 1 full day in minutes

    # Step 6: extract capacities from requests
    Q = {}

    for i in P_nodes:
        Q[i] = R[i][0]

    for i in D_nodes:
        Q[i] = -R[i-n_req][0]

    for i in S_nodes + Z_nodes:
        Q[i] = 0
        
    Q_max = [vehicle[0] for vehicle in K]

    ub_dict = {(i, k): Q_max[k] for i in N for k in bus_idx}

    # Step 7: Create A_m matrix mapping logical nodes to physical stations
    M_stations = list(set(node_to_loc.values()))

    A_m = {}

    # i is the logical node, m is the physical station
    for i in P_and_D:
        for m in M_stations:
            if node_to_loc[i] == m:
                A_m[i, m] = 1
            else:
                A_m[i, m] = 0

    # Step 8: Building model
    # Initializing the model
    model_MILP_base = gp.Model("MILP_base")
    model_MILP_base.Params.OutputFlag = 1

    # Initializing the decision variables
    x_base = model_MILP_base.addVars(N, N, bus_idx, vtype=gp.GRB.BINARY, name="x")
    q_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.INTEGER, lb=0, ub=ub_dict, name="q_k")
    w_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=0, ub=w_max, name="w_k")
    a_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, name="a_k")
    mu = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=1, ub=len(N), name="mu")
    y = model_MILP_base.addVars(P_nodes, vtype=gp.GRB.BINARY, name="y")

    # Objective funtion to be minimized
    obj_expr_trav_cost = gp.quicksum(
        b_1 * bus_cost[k] * t_dict[i, j] * x_base[i, j, k]
        for i in N
        for j in N
        for k in bus_idx
    )

    for i in P_nodes:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in P_and_D for k in bus_idx) == y[i]
        )

    obj_expr_reject_cost = gp.quicksum(
        b_2 * c_0 * (1 - y[i]) for i in P_nodes
    )

    # obj_expr_recourse_cost = b_3 * E

    model_MILP_base.setObjective(obj_expr_trav_cost + obj_expr_reject_cost, gp.GRB.MINIMIZE)

    # Constraints

    # Previous scheduled request served
    for i in P_sched:
        for k in bus_idx:
            model_MILP_base.addConstr(
                gp.quicksum(x_base[i, j, k] for j in N) == u_dict[k, i]
            )

    # Each request served at most once
    for i in P_wait:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N for k in bus_idx) <= 1
        )

    # Ensures same vehicle visits pickup and drop-off nodes of same request
    for i in P_nodes:
        d = i + n_req
        for k in bus_idx:
            model_MILP_base.addConstr(
                gp.quicksum(x_base[i, j, k] for j in N) 
                - gp.quicksum(x_base[d, j, k] for j in N) == 0
            )

    # Flow conservation constraints
    for i in P_nodes + D_nodes:
        for k in bus_idx:
            model_MILP_base.addConstr(
                gp.quicksum(x_base[i, j, k] for j in N) 
                - gp.quicksum(x_base[j, i, k] for j in N) == 0
            )

    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[S_nodes[k], j, k] for j in N) 
            - gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 1
        )

    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[Z_nodes[k], j, k] for j in N) 
            - gp.quicksum(x_base[j, Z_nodes[k], k] for j in N) == -1
        )

    # Capacity constraints
    for i in N:
        for j in N:
            for k in bus_idx:
                model_MILP_base.addConstr(
                    q_k[i, k] + Q[i] - M * (1 - x_base[i, j, k]) <= q_k[j, k]
                )
                model_MILP_base.addConstr(
                    q_k[j, k] <= q_k[i, k] + Q[i] + M * (1 - x_base[i, j, k])
                )

    #for i_idx, i in enumerate(N):
    #    for k in bus_idx:
    #        model_MILP_base.addConstr(
    #            q_k[i_idx, k] <= Q_max[k]
    #        ) ######## Upper bound constraint is already included in decision variable definition

    # Time constraints
    for i in N:
        for j in N:
            for k in bus_idx:
                model_MILP_base.addConstr(
                    a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, j] - M * (1 - x_base[i, j, k]) <= a_k[j, k]
                )

    for m in M_stations:
        model_MILP_base.addConstr(
            gp.quicksum(A_m[i, m] * w_k[i, k] for i in P_and_D for k in bus_idx) <= w_max
        )

    for i in P_nodes:
        for k in bus_idx:
            model_MILP_base.addConstr(
                e_dict[i] <= a_k[i, k] + w_k[i, k] + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
            )
            model_MILP_base.addConstr(
                a_k[i, k] <= l_dict[i] + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
            )

    for i in D_nodes:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i, k] <= l_dict[i] + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
            )

    for i in P_nodes:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i+n_req, k] - a_k[i, k] <= a_max * t_dict[i, i+n_req] 
                + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
            )

    for i in P_nodes:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, i+n_req] <= a_k[i+n_req, k] 
                + M * (1 - gp.quicksum(x_base[i, j, k] for j in N))
            )

    for i in N:
        for j in N:
            for k in bus_idx:
                if i != j and i in P_nodes + D_nodes and j in P_nodes + D_nodes:
                    model_MILP_base.addConstr(
                        mu[i, k] - mu[j, k] + M * x_base[i, j, k] <= M - 1,
                        name=f"subtour_{i}_{j}_{k}"
                    )

    for i in N:
            for k in bus_idx:
                model_MILP_base.addConstr(
                    x_base[i, i, k] == 0
                )

    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 0
        ) # Vehicle cannot enter start node

    for k in bus_idx: # vehicle cannot enter or leave any other vehicle's start or end node
        for k_other in bus_idx:
            if k != k_other:
                # forbid vehicle k from visiting start of other vehicles
                model_MILP_base.addConstr(
                    gp.quicksum(x_base[i, S_nodes[k_other], k] for i in N) == 0
                )
                model_MILP_base.addConstr(
                    gp.quicksum(x_base[S_nodes[k_other], j, k] for j in N) == 0
                )

                # forbid vehicle k from visiting end of other vehicles
                model_MILP_base.addConstr(
                    gp.quicksum(x_base[i, Z_nodes[k_other], k] for i in N) == 0
                )
                model_MILP_base.addConstr(
                    gp.quicksum(x_base[Z_nodes[k_other], j, k] for j in N) == 0
                )
    
    # Step 9: Optimize model
    model_MILP_base.optimize()

    if model_MILP_base.status == gp.GRB.OPTIMAL:
        print("Optimal solution found")

    # Step 10: Updating system state for the next iteration
    if model_MILP_base.status == gp.GRB.OPTIMAL:
        x_sol = model_MILP_base.getAttr('x', x_base)
        a_sol = model_MILP_base.getAttr('x', a_k)
        y_sol = model_MILP_base.getAttr('x', y)

        presented = len(P_wait)
        rejected = sum(1 for i in P_wait if y_sol[i] < 0.5)
        
        history_stats['time_step'].append(t)
        history_stats['new_reqs_presented'].append(presented)
        history_stats['reqs_rejected'].append(rejected)
        history_stats['obj_cost'].append(model_MILP_base.ObjVal)

        for k in bus_idx:
            route_for_k = []
            curr_node = S_nodes[k]
            
            # Follow the active x variables from Start to End node
            while curr_node != Z_nodes[k]:
                # Log current node
                route_for_k.append({
                    'logical_node': curr_node,
                    'location': node_to_loc[curr_node],
                    'arrival_time': a_sol[curr_node, k] if curr_node in a_k else t 
                })
                
                # Find the next node the bus travels to
                next_node = None
                for j in N:
                    if x_sol[curr_node, j, k] > 0.5:
                        next_node = j
                        break
                
                if next_node is not None:
                    curr_node = next_node
                else:
                    break # Safety net in case of broken subtours
            
            # Log the final Z_node
            #route_for_k.append({
            #    'logical_node': curr_node,
            #    'location': node_to_loc[curr_node],
            #    'arrival_time': a_sol[curr_node, k] if curr_node in a_k else t + interval
            #})
            
            history_routes[k].append({
                'interval': t,
                'route': route_for_k
            })

        next_R_sched = []
        next_req_t_p_sched = []
        carried_over_old_P_nodes = []

        # A. Filter Completed & Handle In-Transit Requests
        for idx, i in enumerate(P_nodes):
            d_node = i + n_req
            
            # We check whether request was scheduled in a previous iteration or is newly scheduled in this iteration
            is_active = (i in P_sched) or (i in P_wait and y_sol[i] > 0.5)
            
            if is_active:
                # Find when the drop-off happens
                d_time = -1
                for k in bus_idx:
                    if sum(x_sol[d_node, j, k] for j in N) > 0.5: # Check if bus k visits drop-off node
                        d_time = a_sol[d_node, k] # Extract arrival time at drop-off node
                        break
                
                # If drop-off happens AFTER the current time step, it carries over
                if d_time > t + interval:
                    # Find when and who picked them up
                    p_time = -1
                    assigned_k = None
                    for k in bus_idx:
                        if sum(x_sol[i, j, k] for j in N) > 0.5:
                            p_time = a_sol[i, k]
                            assigned_k = k
                            break
                    
                    carried_req = R[idx].copy() # [pax, orig, dest]
                    carried_time = req_t_p[idx] # Original pickup time request

                    # IN-TRANSIT TRICK: If pickup already happened, move their origin 
                    # to the vehicle's location and set pickup time to now.
                    if p_time <= t + interval:
                        last_visited = S_nodes[assigned_k]
                        max_a = -1
                        for n_idx in N:
                            if sum(x_sol[n_idx, j, assigned_k] for j in N) > 0.5:
                                if a_sol[n_idx, assigned_k] <= t + interval and a_sol[n_idx, assigned_k] > max_a:
                                    max_a = a_sol[n_idx, assigned_k]
                                    last_visited = n_idx
                        
                        # Update origin to vehicle's current node and reset request time
                        carried_req[1] = node_to_loc[last_visited] # Change passenger origin to vehicle's current location
                        carried_time = t + interval # overwrite pickup time to when next simulation starts
                    
                    next_R_sched.append(carried_req)
                    next_req_t_p_sched.append(carried_time)
                    carried_over_old_P_nodes.append(i)

        # B. Update Vehicle Positions (K)
        for k in bus_idx:
            last_visited_node = S_nodes[k]
            max_a = -1
            
            for i in N:
                if i not in Z_nodes:
                    if sum(x_sol[i, j, k] for j in N) > 0.5:
                        if a_sol[i, k] <= t + interval and a_sol[i, k] > max_a:
                            max_a = a_sol[i, k]
                            last_visited_node = i
            
            # Update the physical starting location for the next iteration
            K[k][1] = node_to_loc[last_visited_node]

        # C. Generate New Requests (R_wait)
        new_R_wait = []
        new_req_t_p_wait = []
        
        # Example: Randomly generate 0 to 2 new requests per 10-minute interval
        num_new_requests = random.randint(10, 15) 
        
        for _ in range(num_new_requests):
            pax = random.randint(1, 3)
            orig = random.choice([1, 2, 3, 4, 5])
            dest = random.choice([1, 2, 3, 4, 5])
            while dest == orig:
                dest = random.choice([1, 2, 3, 4, 5])
            
            new_R_wait.append([pax, orig, dest])
            
            # Time of request is randomly distributed in the upcoming interval
            new_req_t_p_wait.append(t + interval + random.randint(0, interval))

        # D. Rebuild u_dict with new logical indices 
        # Next iteration's P_nodes will be ordered as: [ ...new_R_wait..., ...next_R_sched... ]
        u_dict_new = {}
        for k in bus_idx:
            for next_idx, old_i in enumerate(carried_over_old_P_nodes):
                
                # The new index offset must account for the newly added R_wait
                new_P_node_idx = len(new_R_wait) + next_idx 
                
                # Was vehicle k assigned to this request?
                visited_P = sum(x_sol[old_i, j, k] for j in N) > 0.5
                
                if visited_P:
                    u_dict_new[k, new_P_node_idx] = 1
                else:
                    u_dict_new[k, new_P_node_idx] = 0
                    
        # E. Overwrite Globals for Top of Next Loop
        R_wait = new_R_wait
        req_t_p_wait = new_req_t_p_wait
        
        R_sched = next_R_sched
        req_t_p_sched = next_req_t_p_sched
        
        R = R_wait + R_sched
        req_t_p = req_t_p_wait + req_t_p_sched
        n_req = len(R)
        u_dict = u_dict_new

    


Starting iteration for time 0 minutes
Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4464 rows, 1354 columns and 20294 nonzeros (Min)
Model fingerprint: 0x369b17bc
Model has 778 linear objective coefficients and an objective constant of 10000
Variable types: 144 continuous, 1210 integer (1162 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+07]
  Objective range  [3e+01, 1e+03]
  Bounds range     [1e+00, 3e+01]
  RHS range        [1e+00, 1e+07]

Found heuristic solution: objective 10000.000000
Presolve removed 989 rows and 368 columns
Presolve time: 0.07s
Presolved: 3475 rows, 986 columns, 30110 nonzeros
Variable types: 104 continuous, 882 integer (838 binary)

Root relaxation: objective 1.818989e-12, 154 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    

: 

: 

In [ ]:
# --- ADD THIS AT THE VERY END OF YOUR SCRIPT ---
print("\n" + "="*50)
print("SIMULATION COMPLETE - SUMMARY STATISTICS")
print("="*50)

total_reqs = sum(history_stats['new_reqs_presented'])
total_rejected = sum(history_stats['reqs_rejected'])
rejection_rate = (total_rejected / total_reqs * 100) if total_reqs > 0 else 0

print(f"Total Intervals Run: {len(history_stats['time_step'])}")
print(f"Total Requests Presented (excluding initial sched): {total_reqs}")
print(f"Total Requests Rejected: {total_rejected}")
print(f"Overall Rejection Rate: {rejection_rate:.2f}%")
print(f"Average Cost per Interval: {sum(history_stats['obj_cost'])/len(history_stats['obj_cost']):.2f}")

print("\n--- Route Sample ---")
# Loop through ALL buses
for k in bus_idx: 
    print(f"\nBus {k} Routes:")
    
    for iteration in history_routes[k]: 
        t_val = iteration['interval']
        route = iteration['route']
        
        # Format the route into a readable arrow string
        route_str = " -> ".join([f"Loc {stop['location']} (t={stop['arrival_time']:.1f})" for stop in route])
        print(f"  Interval {t_val:3d}: {route_str}")


SIMULATION COMPLETE - SUMMARY STATISTICS
Total Intervals Run: 99
Total Requests Presented (excluding initial sched): 107
Total Requests Rejected: 0
Overall Rejection Rate: 0.00%
Average Cost per Interval: 190.20

--- Route Sample ---

Bus 0 Routes:
  Interval   0: Loc 3 (t=0.0) -> Loc 0 (t=15.0)
  Interval  45: Loc 1 (t=45.0) -> Loc 1 (t=45.0) -> Loc 5 (t=45.0) -> Loc 1 (t=45.0) -> Loc 4 (t=45.0) -> Loc 4 (t=45.0) -> Loc 1 (t=45.0) -> Loc 1 (t=45.0) -> Loc 2 (t=45.0) -> Loc 2 (t=45.0) -> Loc 4 (t=45.0) -> Loc 2 (t=45.0) -> Loc 5 (t=45.0) -> Loc 3 (t=45.0) -> Loc 4 (t=45.0) -> Loc 3 (t=45.0) -> Loc 1 (t=45.0) -> Loc 5 (t=45.0) -> Loc 3 (t=45.0) -> Loc 0 (t=60.0)
  Interval  60: Loc 1 (t=60.0) -> Loc 5 (t=60.0) -> Loc 3 (t=60.0) -> Loc 0 (t=75.0)
  Interval  75: Loc 3 (t=75.0) -> Loc 0 (t=90.0)
  Interval  90: Loc 3 (t=90.0) -> Loc 0 (t=105.0)
  Interval 105: Loc 3 (t=105.0) -> Loc 0 (t=120.0)
  Interval 120: Loc 3 (t=120.0) -> Loc 3 (t=120.0) -> Loc 2 (t=120.0) -> Loc 0 (t=135.0)
  Int

In [ ]:
STOP

NameError: name 'STOP' is not defined

: 

# OLD

In [ ]:
# Requests considered (passengers, origin, destination)
r_1 = [2, 1, 5]
r_2 = [4, 2, 4]
r_3 = [1, 1, 4]
r_4 = [3, 2, 5]
r_5 = [2, 1, 4]
r_6 = [1, 5, 3]
r_7 = [4, 4, 2]
r_8 = [2, 3, 1]
r_9 = [3, 3, 4]
r_10 = [1, 5, 1]
R_wait = [r_1, r_2, r_3, r_4, r_5, r_6, r_7, r_8, r_9, r_10]
R_sched = []
R_pred = []
R = R_wait + R_sched

n_req = len(R)

bus_idx = list(range(0, 2))

# Vehicles (maximum capacity, current position, virtual destination)
k_1 = [8, 3, 0]
k_2 = [6, 5, 0]
K = [k_1, k_2]
bus_idx = list(range(0, len(K)))

bus_cost = {0: 40, 1: 30}
reject_cost = 100

P_nodes = list(range(n_req))
D_nodes = list(range(n_req, 2*n_req))
S_nodes = list(range(2*n_req, 2*n_req + len(K)))
Z_nodes = list(range(2*n_req + len(K), 2*n_req + 2*len(K)))
N = P_nodes + D_nodes + S_nodes + Z_nodes

P_loc = {i: R[i][1] for i in range(n_req)}
D_loc = {i+n_req: R[i][2] for i in range(n_req)}
S_loc = {S_nodes[k]: K[k][1] for k in range(len(K))}
Z_loc = {Z_nodes[k]: K[k][2] for k in range(len(K))}

node_to_loc = {}
node_to_loc.update(P_loc)
node_to_loc.update(D_loc)
node_to_loc.update(S_loc)
node_to_loc.update(Z_loc)

s = np.array([1, 1, 1, 1, 1])

t = np.array([[0, 2, 3, 2, 4],
     [3, 0, 4, 7, 2],
     [2, 3, 0, 3, 4],
     [2, 6, 1, 0, 3],
     [5, 4, 5, 2, 0]])

node_to_idx = {1:0, 2:1, 3:2, 4:3, 5:4}

# Create dictionary travel times keyed by logical nodes
t_dict = {(i,j): t[node_to_idx[node_to_loc[i]], node_to_idx[node_to_loc[j]]] for i in N for j in N}

s_dict = {i: s[node_to_idx[node_to_loc[i]]] for i in N}

w_max = 10

Q = {}

for i in P_nodes:
    Q[i] = R[i][0]

for i in D_nodes:
    Q[i] = -R[i-n_req][0]

for i in S_nodes + Z_nodes:
    Q[i] = 0
    
Q_max = [K[0] for K in K]

ub_dict = {(i, k): Q_max[k] for i in N for k in bus_idx}

# Your logical node numbers
nodes_logical = [1,2,3,4,5]

u = np.zeros((len(K), len(nodes_logical)))

# Map logical numbers to 0-based matrix indices
node_to_idx = {1:0, 2:1, 3:2, 4:3, 5:4}

# Create dictionary travel times keyed by logical nodes
u_dict = {(k,i): u[k, node_to_idx[i]] for i in nodes_logical for k in bus_idx}

# Initializing the model
model_MILP_base = gp.Model("MILP_base")
model_MILP_base.Params.OutputFlag = 1

# Initializing the decision variables
x_base = model_MILP_base.addVars(N, N, bus_idx, vtype=gp.GRB.BINARY, name="x")
q_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.INTEGER, lb=0, ub=ub_dict, name="q_k")
w_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=0, ub=w_max, name="w_k")
a_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, name="a_k")

# Objective funtion to be minimized
obj_expr_trav_cost = gp.quicksum(
    b_1 * bus_cost[k] * t_dict[i, j] * x_base[i, j, k]
    for i in N
    for j in N
    for k in bus_idx
)

obj_expr_reject_cost = gp.quicksum(
    b_2 * c_0 * (1 - gp.quicksum(x_base[i, j, k] for k in bus_idx for j in P_and_D))
    for i in P_nodes
)

# obj_expr_recourse_cost = b_3 * E

model_MILP_base.setObjective(obj_expr_trav_cost + obj_expr_reject_cost, gp.GRB.MINIMIZE)

# Constraints

# Previous scheduled request served
for i in P_sched:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) == u_dict[k, i]
        )

# Each request served at most once
for i in P_wait:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) <= 1
        )

# Ensures same vehicle visits pickup and drop-off nodes of same request
for i in P_nodes:
    d = i + n_req
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[d, j, k] for j in N) == 0
        )

# Flow conservation constraints
for i in P_nodes + D_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[j, i, k] for j in N) == 0
        )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[S_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 1
    )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[Z_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, Z_nodes[k], k] for j in N) == -1
    )

# Capacity constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                q_k[i, k] + Q[i] - M * (1 - x_base[i, j, k]) <= q_k[j, k]
            )

#for i_idx, i in enumerate(N):
#    for k in bus_idx:
#        model_MILP_base.addConstr(
#            q_k[i_idx, k] <= Q_max[k]
#        ) ######## Upper bound constraint is already included in decision variable definition

# Time constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, j] - M * (1 - x_base[i, j, k]) <= a_k[j, k]
            )


# OLD 2

In [ ]:
# Initializing the model
model_MILP_base = gp.Model("MILP_base")
model_MILP_base.Params.OutputFlag = 1

# Initializing the decision variables
x_base = model_MILP_base.addVars(N, N, bus_idx, vtype=gp.GRB.BINARY, name="x")
q_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.INTEGER, lb=0, ub=ub_dict, name="q_k")
w_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=0, ub=w_max, name="w_k")
a_k = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, name="a_k")
mu = model_MILP_base.addVars(N, bus_idx, vtype=gp.GRB.CONTINUOUS, lb=1, ub=len(N), name="mu")

# Objective funtion to be minimized
obj_expr_trav_cost = gp.quicksum(
    b_1 * bus_cost[k] * t_dict[i, j] * x_base[i, j, k]
    for i in N
    for j in N
    for k in bus_idx
)

obj_expr_reject_cost = gp.quicksum(
    b_2 * c_0 * (1 - gp.quicksum(x_base[i, j, k] for k in bus_idx for j in P_and_D))
    for i in P_nodes
)

# obj_expr_recourse_cost = b_3 * E

model_MILP_base.setObjective(obj_expr_trav_cost + obj_expr_reject_cost, gp.GRB.MINIMIZE)

# Constraints

# Previous scheduled request served
for i in P_sched:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) == u_dict[k, i]
        )

# Each request served at most once
for i in P_wait:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) <= 1
        )

# Ensures same vehicle visits pickup and drop-off nodes of same request
for i in P_nodes:
    d = i + n_req
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[d, j, k] for j in N) == 0
        )

# Flow conservation constraints
for i in P_nodes + D_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            gp.quicksum(x_base[i, j, k] for j in N) 
            - gp.quicksum(x_base[j, i, k] for j in N) == 0
        )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[S_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, S_nodes[k], k] for j in N) == 1
    )

for k in bus_idx:
    model_MILP_base.addConstr(
        gp.quicksum(x_base[Z_nodes[k], j, k] for j in N) 
        - gp.quicksum(x_base[j, Z_nodes[k], k] for j in N) == -1
    )

# Capacity constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                q_k[i, k] + Q[i] - M * (1 - x_base[i, j, k]) <= q_k[j, k]
            )

#for i_idx, i in enumerate(N):
#    for k in bus_idx:
#        model_MILP_base.addConstr(
#            q_k[i_idx, k] <= Q_max[k]
#        ) ######## Upper bound constraint is already included in decision variable definition

# Time constraints
for i in N:
    for j in N:
        for k in bus_idx:
            model_MILP_base.addConstr(
                a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, j] - M * (1 - x_base[i, j, k]) <= a_k[j, k]
            )

for k in bus_idx:
    for m in M_stations:
        model_MILP_base.addConstr(
            gp.quicksum(A_m[i, m] * w_k[i, k] for i in P_and_D) <= w_max
        )

for i in P_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            e_dict[i] <= a_k[i, k] + w_k[i, k]
        )
        model_MILP_base.addConstr(
            a_k[i, k] <= l_dict[i]
        )

for i in D_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i, k] <= l_dict[i]
        )

for i in P_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i+n_req, k] - a_k[i, k] <= a_max * t_dict[i, i+n_req]
        )

for i in P_nodes:
    for k in bus_idx:
        model_MILP_base.addConstr(
            a_k[i, k] + w_k[i, k] + s_dict[i] + t_dict[i, i+n_req] <= a_k[i+n_req, k]
        )

for i in N:
    for j in N:
        for k in bus_idx:
            if i != j and i != 0 and j != 0:
                model_MILP_base.addConstr(
                    mu[i, k] - mu[j, k] + M * x_base[i, j, k] <= M - 1,
                    name=f"subtour_{i}_{j}_{k}"
                )

for i in N:
        for k in bus_idx:
            if i == j:
                model_MILP_base.addConstr(
                    x_base[i, i, k] == 0
                )